<a href="https://colab.research.google.com/github/sonjoy1s/Rnn_DeepLearning/blob/main/RNN_LSTMModule_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install nltk

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk
from sklearn.model_selection import train_test_split

In [4]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
document = """Reading short stories in English is a great way to improve your language level. In this section, read our short stories that were specially written for English language learners. There are two sections, one for lower level learners (A2/B1) and one for higher levels (B2/C1).

You will improve your reading fluency and comprehension and develop your vocabulary. Each story has interactive exercises to help you understand and use the language."""

# Tokenize

In [6]:
tokens = nltk.word_tokenize(document.lower())
print(tokens)

['reading', 'short', 'stories', 'in', 'english', 'is', 'a', 'great', 'way', 'to', 'improve', 'your', 'language', 'level', '.', 'in', 'this', 'section', ',', 'read', 'our', 'short', 'stories', 'that', 'were', 'specially', 'written', 'for', 'english', 'language', 'learners', '.', 'there', 'are', 'two', 'sections', ',', 'one', 'for', 'lower', 'level', 'learners', '(', 'a2/b1', ')', 'and', 'one', 'for', 'higher', 'levels', '(', 'b2/c1', ')', '.', 'you', 'will', 'improve', 'your', 'reading', 'fluency', 'and', 'comprehension', 'and', 'develop', 'your', 'vocabulary', '.', 'each', 'story', 'has', 'interactive', 'exercises', 'to', 'help', 'you', 'understand', 'and', 'use', 'the', 'language', '.']


# Build Vocabulary

In [7]:
#build vocab
vocab = {'<unk>':0}

In [8]:
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)


In [9]:
vocab

{'<unk>': 0,
 'reading': 1,
 'short': 2,
 'stories': 3,
 'in': 4,
 'english': 5,
 'is': 6,
 'a': 7,
 'great': 8,
 'way': 9,
 'to': 10,
 'improve': 11,
 'your': 12,
 'language': 13,
 'level': 14,
 '.': 15,
 'this': 16,
 'section': 17,
 ',': 18,
 'read': 19,
 'our': 20,
 'that': 21,
 'were': 22,
 'specially': 23,
 'written': 24,
 'for': 25,
 'learners': 26,
 'there': 27,
 'are': 28,
 'two': 29,
 'sections': 30,
 'one': 31,
 'lower': 32,
 '(': 33,
 'a2/b1': 34,
 ')': 35,
 'and': 36,
 'higher': 37,
 'levels': 38,
 'b2/c1': 39,
 'you': 40,
 'will': 41,
 'fluency': 42,
 'comprehension': 43,
 'develop': 44,
 'vocabulary': 45,
 'each': 46,
 'story': 47,
 'has': 48,
 'interactive': 49,
 'exercises': 50,
 'help': 51,
 'understand': 52,
 'use': 53,
 'the': 54}

In [10]:
len(vocab)

55

# Extract Sentence from data

In [11]:
input_sentance = document.split('\n')

In [12]:
def text_to_indexs(sentence, vocab):
  numerical_sentence = []
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence

In [13]:
input_numerical_sentence = []
for sentence in input_sentance:
  input_numerical_sentence.append(text_to_indexs(word_tokenize(sentence.lower()),vocab))

In [14]:
input_numerical_sentence

[[1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  4,
  16,
  17,
  18,
  19,
  20,
  2,
  3,
  21,
  22,
  23,
  24,
  25,
  5,
  13,
  26,
  15,
  27,
  28,
  29,
  30,
  18,
  31,
  25,
  32,
  14,
  26,
  33,
  34,
  35,
  36,
  31,
  25,
  37,
  38,
  33,
  39,
  35,
  15],
 [],
 [40,
  41,
  11,
  12,
  1,
  42,
  36,
  43,
  36,
  44,
  12,
  45,
  15,
  46,
  47,
  48,
  49,
  50,
  10,
  51,
  40,
  52,
  36,
  53,
  54,
  13,
  15]]

# Training sequence form

In [15]:
training_sequence = []
for sentence in input_numerical_sentence:
  for i in range(1,len(sentence)):
    training_sequence.append(sentence[:i+1])

In [16]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]

In [17]:
sequence_length = []
for sequence in training_sequence:
  sequence_length.append(len(sequence))


In [18]:
max(sequence_length)

54

In [19]:
[0] * (max(sequence_length)-len(training_sequence[0]))

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [21]:
padded_training_sequence = []
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(sequence_length) - len(sequence))+sequence)

In [22]:
len(padded_training_sequence[0])


54

In [23]:
padded_training_sequence[0]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2]

In [24]:
padded_training_sequence[6]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8]

In [25]:
padded_training_sequence = torch.tensor(padded_training_sequence,dtype=torch.long)

In [26]:
padded_training_sequence

tensor([[ 0,  0,  0,  ...,  0,  1,  2],
        [ 0,  0,  0,  ...,  1,  2,  3],
        [ 0,  0,  0,  ...,  2,  3,  4],
        ...,
        [ 0,  0,  0,  ..., 36, 53, 54],
        [ 0,  0,  0,  ..., 53, 54, 13],
        [ 0,  0,  0,  ..., 54, 13, 15]])

In [27]:
X = padded_training_sequence[:, :-1]

In [28]:
X

tensor([[ 0,  0,  0,  ...,  0,  0,  1],
        [ 0,  0,  0,  ...,  0,  1,  2],
        [ 0,  0,  0,  ...,  1,  2,  3],
        ...,
        [ 0,  0,  0,  ..., 52, 36, 53],
        [ 0,  0,  0,  ..., 36, 53, 54],
        [ 0,  0,  0,  ..., 53, 54, 13]])

In [29]:
y = padded_training_sequence[:,-1]

In [30]:
y

tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  4, 16, 17, 18,
        19, 20,  2,  3, 21, 22, 23, 24, 25,  5, 13, 26, 15, 27, 28, 29, 30, 18,
        31, 25, 32, 14, 26, 33, 34, 35, 36, 31, 25, 37, 38, 33, 39, 35, 15, 41,
        11, 12,  1, 42, 36, 43, 36, 44, 12, 45, 15, 46, 47, 48, 49, 50, 10, 51,
        40, 52, 36, 53, 54, 13, 15])

In [31]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx) :
    return self.X[idx], self.y[idx]


In [33]:
dataset = CustomDataset(X,y)


In [34]:
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

In [35]:
for input , output in dataloader:
  print(input,output)

tensor([[ 0,  0,  0,  ..., 12,  1, 42],
        [ 0,  0,  0,  ..., 22, 23, 24],
        [ 0,  0,  0,  ..., 36, 43, 36],
        ...,
        [ 0,  0,  0,  ..., 14, 15,  4],
        [ 0,  0,  0,  ..., 12, 45, 15],
        [ 0,  0,  0,  ...,  5, 13, 26]]) tensor([36, 25, 44,  2, 27,  5, 50, 19, 54, 13, 15, 24, 11, 36, 12, 35,  5, 14,
         6, 26,  2,  4, 41, 43, 35, 47, 52, 28, 31, 16, 46, 15])
tensor([[ 0,  0,  0,  ...,  4,  5,  6],
        [ 0,  0,  0,  ..., 28, 29, 30],
        [ 0,  0,  0,  ..., 33, 34, 35],
        ...,
        [ 0,  0,  0,  ..., 12, 13, 14],
        [ 0,  0,  0,  ...,  4, 16, 17],
        [ 0,  0,  0,  ..., 31, 25, 37]]) tensor([ 7, 18, 36, 12, 11, 17, 25, 12, 29,  8,  3, 45, 53,  9, 10, 31, 13, 26,
        21, 20, 33, 14, 37,  4, 10, 39, 49, 40, 36, 15, 18, 38])
tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,

In [36]:
X.shape

torch.Size([79, 53])

In [37]:
y.shape

torch.Size([79])

In [38]:
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,100)
    self.lstm = nn.LSTM(100,150,batch_first=True)
    self.fc = nn.Linear(150,vocab_size)


  def forward():
    pass